# Content-based & hybrid recommenders — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x)))
def dcg(rels): return sum((2**r-1)/math.log2(i+2) for i,r in enumerate(rels))
def ndcg(rels):
    best=sorted(rels, reverse=True)
    return dcg(rels)/dcg(best) if dcg(best)>0 else 0.0
print('setup ok')

## Using item attributes, then blending them with behavior
Content recommenders score item features against a user profile; hybrids mix that content signal with collaborative evidence. The examples compute a profile, score candidates, blend two signals, tune the blend, and handle a new item.

In [ ]:
# 1) content profile is the rating-weighted average of liked item features
X=np.array([[1,0,1],[1,1,0],[0,1,1],[0,0,1]],float); ratings=np.array([5,4,1,0],float)
profile=ratings@X/ratings.sum()
plt.figure(figsize=(6,3)); plt.bar(['visual','text','video'],profile); plt.title('user content profile')
assert np.allclose(profile,[0.9,0.5,0.6])

In [ ]:
# 2) cosine content scores rank candidate items by feature match
profile=np.array([0.9,0.5,0.6])
C=np.array([[1,0,0],[0,1,1],[1,1,0],[0,0,1]],float)
scores=np.array([cosine(profile,c) for c in C])
plt.figure(figsize=(6,3)); plt.bar([f'cand {i}' for i in range(4)],scores); plt.title('content cosine scores')
assert int(np.argmax(scores))==2 and abs(scores[2]-0.8307471607356973)<1e-9

In [ ]:
# 3) hybrid score blends content and collaborative scores
content=np.array([0.79,0.74,0.85]); collab=np.array([0.30,0.90,0.70]); alpha=.4
hybrid=alpha*content+(1-alpha)*collab
plt.figure(figsize=(6,3)); plt.bar(['A','B','C'],hybrid); plt.title('alpha=.4 hybrid score')
assert int(np.argmax(hybrid))==1 and abs(hybrid[1]-0.836)<1e-12

In [ ]:
# 4) the blend knob changes which item wins
content=np.array([0.79,0.74,0.85]); collab=np.array([0.30,0.90,0.70])
alphas=np.linspace(0,1,101); scores=np.array([a*content+(1-a)*collab for a in alphas])
winners=np.argmax(scores,axis=1)
plt.figure(figsize=(6,3)); plt.plot(alphas,scores); plt.xlabel('alpha content weight'); plt.ylabel('score'); plt.title('hybrid tradeoff')
assert winners[0]==1 and winners[-1]==2

In [ ]:
# 5) cold new item: collaborative score missing, content still works
content=np.array([0.79,0.74,0.85,0.82]); collab=np.array([0.30,0.90,0.70,np.nan]); alpha=.4
hybrid=np.where(np.isnan(collab), content, alpha*content+(1-alpha)*collab)
plt.figure(figsize=(6,3)); plt.bar(['A','B','C','new'],hybrid); plt.title('content rescues a new item')
assert not np.isnan(hybrid[-1]) and abs(hybrid[-1]-0.82)<1e-12